# 001 — Clustering Comparison

Head-to-head comparison of five clustering algorithms: **K-Means**, **DBSCAN**, **HDBSCAN**,
**Gaussian Mixture Model (GMM)**, and **Agglomerative Clustering**.

Covers parameter search, internal validation metrics (Silhouette, Davies-Bouldin,
Calinski-Harabasz), UMAP and PCA 2D visualizations, and a ranked comparison across
all methods.

**Lifecycle stage:** seedling (model-garden)

All code is self-contained — no external imports from a shared `src/` package.

In [ ]:
# ---------------------------------------------------------------------------
# Papermill parameters  (this cell is tagged "parameters")
# ---------------------------------------------------------------------------

# Data loading
feature_paths: list[str] = []          # local or gs:// URIs; empty -> synthetic
feature_cols: list[str] | None = None  # subset of columns; None -> all

# Clustering search ranges
n_clusters_range: list[int] = [2, 3, 4, 5, 6, 7, 8]
dbscan_eps_range: list[float] = [0.3, 0.5, 0.7, 1.0, 1.5]
dbscan_min_samples: int = 5

# Reproducibility
random_state: int = 42

# UMAP
umap_n_neighbors: int = 15

# Outputs
metrics_json_path: str = "outputs/metrics/metrics.json"
plots_dir: str = "outputs/plots"
executed_notebook_path: str | None = None

In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import json
import os
import warnings
from datetime import datetime, timezone
from pathlib import Path

warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from sklearn.cluster import KMeans, DBSCAN, HDBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA
from sklearn.datasets import make_blobs

import umap

# ---------------------------------------------------------------------------
# Output directories
# ---------------------------------------------------------------------------
Path(plots_dir).mkdir(parents=True, exist_ok=True)
Path(metrics_json_path).parent.mkdir(parents=True, exist_ok=True)

RUN_START = datetime.now(timezone.utc)
print(f"Run started: {RUN_START.isoformat()}")
print(f"plots_dir          : {plots_dir}")
print(f"metrics_json_path  : {metrics_json_path}")

## 1 — Data Loading

In [ ]:
# ---------------------------------------------------------------------------
# Data loading
# ---------------------------------------------------------------------------

def _make_synthetic(random_state: int = 42) -> pl.DataFrame:
    """Generate 2000-sample, 10-feature dataset with 5 anisotropic Gaussian clusters."""
    rng = np.random.default_rng(random_state)

    X_raw, y_true = make_blobs(
        n_samples=2000,
        n_features=10,
        centers=5,
        cluster_std=1.5,
        random_state=random_state,
    )

    # Random rotation to make clusters anisotropic
    rotation, _ = np.linalg.qr(rng.standard_normal((10, 10)))
    X_rot = X_raw @ rotation

    cols = {f"feature_{i}": X_rot[:, i] for i in range(X_rot.shape[1])}
    cols["true_label"] = y_true.astype(np.int32)
    return pl.DataFrame(cols)


def _load_paths(paths: list[str], cols: list[str] | None) -> pl.DataFrame:
    """Load parquet or CSV files into a single Polars DataFrame."""
    frames = []
    for p in paths:
        if p.endswith(".csv"):
            df = pl.read_csv(p)
        else:
            df = pl.read_parquet(p)
        if cols is not None:
            df = df.select(cols)
        frames.append(df)
    return pl.concat(frames)


if len(feature_paths) == 0:
    print("No feature_paths provided — generating synthetic dataset.")
    df = _make_synthetic(random_state=random_state)
    synthetic = True
else:
    print(f"Loading {len(feature_paths)} path(s)...")
    df = _load_paths(feature_paths, feature_cols)
    synthetic = False

print(f"\nDataset shape: {df.shape}")
print(df.head(3))

## 2 — EDA & Dimensionality Reduction

In [ ]:
# ---------------------------------------------------------------------------
# Schema & summary statistics
# ---------------------------------------------------------------------------
print("Schema:")
print(df.schema)
print("\nDescriptive statistics:")
print(df.describe())

# Select numeric feature columns (drop any label columns present)
feature_col_names = [
    c for c in df.columns
    if c not in ("true_label",)
    and df[c].dtype in (pl.Float32, pl.Float64, pl.Int32, pl.Int64, pl.UInt32, pl.UInt64)
]
if feature_cols is not None:
    feature_col_names = [c for c in feature_col_names if c in feature_cols]

print(f"\nFeature columns ({len(feature_col_names)}): {feature_col_names}")

# Convert to numpy and scale
X_raw = df.select(feature_col_names).to_numpy().astype(np.float64)
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print(f"\nX shape after scaling: {X.shape}")

In [ ]:
# ---------------------------------------------------------------------------
# PCA 2D projection (EDA)
# ---------------------------------------------------------------------------
pca = PCA(n_components=2, random_state=random_state)
embedding_pca = pca.fit_transform(X)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(embedding_pca[:, 0], embedding_pca[:, 1],
           c='grey', s=8, alpha=0.4, linewidths=0)
ax.set_title("PCA 2D Projection (no labels)")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} var)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} var)")
fig.tight_layout()
_pca_eda_path = Path(plots_dir) / "eda_pca.png"
fig.savefig(_pca_eda_path, dpi=120)
plt.show()
print(f"PCA explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Saved: {_pca_eda_path}")

In [ ]:
# ---------------------------------------------------------------------------
# UMAP 2D projection (EDA)
# ---------------------------------------------------------------------------
print("Fitting UMAP...")
umap_reducer = umap.UMAP(
    n_components=2,
    n_neighbors=umap_n_neighbors,
    random_state=random_state,
    low_memory=False,
)
embedding_umap = umap_reducer.fit_transform(X)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(embedding_umap[:, 0], embedding_umap[:, 1],
           c='grey', s=8, alpha=0.4, linewidths=0)
ax.set_title("UMAP 2D Projection (no labels)")
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
fig.tight_layout()
_umap_eda_path = Path(plots_dir) / "eda_umap.png"
fig.savefig(_umap_eda_path, dpi=120)
plt.show()
print(f"Saved: {_umap_eda_path}")

## 3 — Clustering Algorithms

In [ ]:
# ---------------------------------------------------------------------------
# Helper: compute internal metrics safely
# ---------------------------------------------------------------------------

def _compute_metrics(X: np.ndarray, labels: np.ndarray) -> dict:
    """Return silhouette, davies_bouldin, calinski_harabasz.  Returns NaN sentinels
    when fewer than 2 clusters or all points are noise."""
    unique = np.unique(labels[labels != -1])
    n_clusters = len(unique)
    if n_clusters < 2:
        return {"silhouette": float('nan'),
                "davies_bouldin": float('nan'),
                "calinski_harabasz": float('nan')}
    mask = labels != -1
    X_valid = X[mask]
    labels_valid = labels[mask]
    if len(np.unique(labels_valid)) < 2:
        return {"silhouette": float('nan'),
                "davies_bouldin": float('nan'),
                "calinski_harabasz": float('nan')}
    return {
        "silhouette": float(silhouette_score(X_valid, labels_valid, sample_size=min(5000, len(X_valid)), random_state=random_state)),
        "davies_bouldin": float(davies_bouldin_score(X_valid, labels_valid)),
        "calinski_harabasz": float(calinski_harabasz_score(X_valid, labels_valid)),
    }

In [ ]:
# ---------------------------------------------------------------------------
# K-Means parameter search
# ---------------------------------------------------------------------------
print("=" * 60)
print("K-MEANS")
print("=" * 60)

kmeans_results = []
for k in n_clusters_range:
    km = KMeans(n_clusters=k, random_state=random_state, n_init=10)
    labels_km = km.fit_predict(X)
    metrics = _compute_metrics(X, labels_km)
    kmeans_results.append({
        "k": k,
        "inertia": float(km.inertia_),
        "labels": labels_km,
        **metrics,
    })
    print(f"  k={k:2d}  inertia={km.inertia_:10.1f}  "
          f"sil={metrics['silhouette']:.4f}  "
          f"db={metrics['davies_bouldin']:.4f}  "
          f"ch={metrics['calinski_harabasz']:.1f}")

# Best by silhouette
valid_km = [r for r in kmeans_results if not np.isnan(r['silhouette'])]
best_km = max(valid_km, key=lambda r: r['silhouette'])
print(f"\n  Best k (by silhouette): k={best_km['k']}  silhouette={best_km['silhouette']:.4f}")

In [ ]:
# ---------------------------------------------------------------------------
# K-Means elbow & silhouette plots
# ---------------------------------------------------------------------------
ks = [r['k'] for r in kmeans_results]
inertias = [r['inertia'] for r in kmeans_results]
sils = [r['silhouette'] for r in kmeans_results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(ks, inertias, marker='o', color='steelblue')
ax1.axvline(best_km['k'], color='tomato', linestyle='--', label=f"Best k={best_km['k']}")
ax1.set_title("K-Means Elbow (Inertia)")
ax1.set_xlabel("k")
ax1.set_ylabel("Inertia")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(ks, sils, marker='s', color='seagreen')
ax2.axvline(best_km['k'], color='tomato', linestyle='--', label=f"Best k={best_km['k']}")
ax2.set_title("K-Means Silhouette Score")
ax2.set_xlabel("k")
ax2.set_ylabel("Silhouette")
ax2.legend()
ax2.grid(True, alpha=0.3)

fig.tight_layout()
_km_path = Path(plots_dir) / "kmeans_elbow.png"
fig.savefig(_km_path, dpi=120)
plt.show()
print(f"Saved: {_km_path}")

In [ ]:
# ---------------------------------------------------------------------------
# DBSCAN parameter search
# ---------------------------------------------------------------------------
print("=" * 60)
print("DBSCAN")
print("=" * 60)

dbscan_results = []
for eps in dbscan_eps_range:
    db = DBSCAN(eps=eps, min_samples=dbscan_min_samples)
    labels_db = db.fit_predict(X)
    n_clusters_db = len(set(labels_db) - {-1})
    n_noise = int((labels_db == -1).sum())
    if n_clusters_db < 2:
        print(f"  eps={eps:.2f}  n_clusters={n_clusters_db}  n_noise={n_noise}  -> skipped (< 2 clusters)")
        dbscan_results.append({
            "eps": eps,
            "n_clusters": n_clusters_db,
            "n_noise": n_noise,
            "labels": labels_db,
            "silhouette": float('nan'),
            "davies_bouldin": float('nan'),
            "calinski_harabasz": float('nan'),
        })
        continue
    metrics = _compute_metrics(X, labels_db)
    dbscan_results.append({
        "eps": eps,
        "n_clusters": n_clusters_db,
        "n_noise": n_noise,
        "labels": labels_db,
        **metrics,
    })
    print(f"  eps={eps:.2f}  n_clusters={n_clusters_db:2d}  n_noise={n_noise:4d}  "
          f"sil={metrics['silhouette']:.4f}  "
          f"db={metrics['davies_bouldin']:.4f}")

# Best by silhouette
valid_db = [r for r in dbscan_results if not np.isnan(r['silhouette'])]
if valid_db:
    best_db = max(valid_db, key=lambda r: r['silhouette'])
    print(f"\n  Best eps (by silhouette): eps={best_db['eps']}  silhouette={best_db['silhouette']:.4f}")
else:
    # Fallback: pick the eps that yielded the most clusters
    best_db = max(dbscan_results, key=lambda r: r['n_clusters'])
    print("  No valid DBSCAN configuration found (all skipped); using fallback.")

In [ ]:
# ---------------------------------------------------------------------------
# HDBSCAN
# ---------------------------------------------------------------------------
print("=" * 60)
print("HDBSCAN")
print("=" * 60)

hdbscan_min_cluster_size = max(5, len(X) // 50)
print(f"  min_cluster_size={hdbscan_min_cluster_size}  min_samples={dbscan_min_samples}")

hdb = HDBSCAN(min_cluster_size=hdbscan_min_cluster_size, min_samples=dbscan_min_samples)
labels_hdb = hdb.fit_predict(X)
n_clusters_hdb = len(set(labels_hdb) - {-1})
n_noise_hdb = int((labels_hdb == -1).sum())

print(f"  n_clusters={n_clusters_hdb}  n_noise={n_noise_hdb}")

if n_clusters_hdb >= 2:
    metrics_hdb = _compute_metrics(X, labels_hdb)
else:
    metrics_hdb = {"silhouette": float('nan'), "davies_bouldin": float('nan'), "calinski_harabasz": float('nan')}

hdbscan_result = {
    "min_cluster_size": hdbscan_min_cluster_size,
    "min_samples": dbscan_min_samples,
    "n_clusters": n_clusters_hdb,
    "n_noise": n_noise_hdb,
    "labels": labels_hdb,
    **metrics_hdb,
}
print(f"  silhouette={metrics_hdb['silhouette']:.4f}  "
      f"db={metrics_hdb['davies_bouldin']:.4f}  "
      f"ch={metrics_hdb['calinski_harabasz']:.1f}")

In [ ]:
# ---------------------------------------------------------------------------
# Gaussian Mixture Model parameter search
# ---------------------------------------------------------------------------
print("=" * 60)
print("GAUSSIAN MIXTURE MODEL")
print("=" * 60)

gmm_results = []
for k in n_clusters_range:
    gmm = GaussianMixture(n_components=k, random_state=random_state, n_init=3)
    gmm.fit(X)
    labels_gmm = gmm.predict(X)
    bic = float(gmm.bic(X))
    aic = float(gmm.aic(X))
    metrics = _compute_metrics(X, labels_gmm)
    gmm_results.append({
        "k": k,
        "bic": bic,
        "aic": aic,
        "labels": labels_gmm,
        **metrics,
    })
    print(f"  k={k:2d}  bic={bic:12.1f}  aic={aic:12.1f}  "
          f"sil={metrics['silhouette']:.4f}  "
          f"db={metrics['davies_bouldin']:.4f}")

# Best by BIC (lower is better)
best_gmm = min(gmm_results, key=lambda r: r['bic'])
print(f"\n  Best k (by BIC): k={best_gmm['k']}  bic={best_gmm['bic']:.1f}  silhouette={best_gmm['silhouette']:.4f}")

In [ ]:
# ---------------------------------------------------------------------------
# Agglomerative Clustering parameter search
# ---------------------------------------------------------------------------
print("=" * 60)
print("AGGLOMERATIVE CLUSTERING (Ward linkage)")
print("=" * 60)

agg_results = []
for k in n_clusters_range:
    agg = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels_agg = agg.fit_predict(X)
    metrics = _compute_metrics(X, labels_agg)
    agg_results.append({
        "k": k,
        "labels": labels_agg,
        **metrics,
    })
    print(f"  k={k:2d}  sil={metrics['silhouette']:.4f}  "
          f"db={metrics['davies_bouldin']:.4f}  "
          f"ch={metrics['calinski_harabasz']:.1f}")

# Best by silhouette
valid_agg = [r for r in agg_results if not np.isnan(r['silhouette'])]
best_agg = max(valid_agg, key=lambda r: r['silhouette'])
print(f"\n  Best k (by silhouette): k={best_agg['k']}  silhouette={best_agg['silhouette']:.4f}")

## 4 — Evaluation & Comparison

In [ ]:
# ---------------------------------------------------------------------------
# Build results summary
# ---------------------------------------------------------------------------
algorithm_results = {
    "KMeans": {
        "best_params": {"k": best_km['k']},
        "n_clusters": best_km['k'],
        "silhouette": best_km['silhouette'],
        "davies_bouldin": best_km['davies_bouldin'],
        "calinski_harabasz": best_km['calinski_harabasz'],
        "labels": best_km['labels'],
    },
    "DBSCAN": {
        "best_params": {"eps": best_db['eps'], "min_samples": dbscan_min_samples},
        "n_clusters": best_db['n_clusters'],
        "silhouette": best_db['silhouette'],
        "davies_bouldin": best_db['davies_bouldin'],
        "calinski_harabasz": best_db['calinski_harabasz'],
        "labels": best_db['labels'],
    },
    "HDBSCAN": {
        "best_params": {"min_cluster_size": hdbscan_result['min_cluster_size'], "min_samples": hdbscan_result['min_samples']},
        "n_clusters": hdbscan_result['n_clusters'],
        "silhouette": hdbscan_result['silhouette'],
        "davies_bouldin": hdbscan_result['davies_bouldin'],
        "calinski_harabasz": hdbscan_result['calinski_harabasz'],
        "labels": hdbscan_result['labels'],
    },
    "GMM": {
        "best_params": {"k": best_gmm['k']},
        "n_clusters": best_gmm['k'],
        "silhouette": best_gmm['silhouette'],
        "davies_bouldin": best_gmm['davies_bouldin'],
        "calinski_harabasz": best_gmm['calinski_harabasz'],
        "labels": best_gmm['labels'],
    },
    "Agglomerative": {
        "best_params": {"k": best_agg['k'], "linkage": "ward"},
        "n_clusters": best_agg['k'],
        "silhouette": best_agg['silhouette'],
        "davies_bouldin": best_agg['davies_bouldin'],
        "calinski_harabasz": best_agg['calinski_harabasz'],
        "labels": best_agg['labels'],
    },
}

# Rank by silhouette (NaN last)
def _sil_key(item):
    v = item[1]['silhouette']
    return v if not np.isnan(v) else -999.0

ranked = sorted(algorithm_results.items(), key=_sil_key, reverse=True)

print(f"{'Algorithm':<18} {'n_clusters':>10} {'Silhouette':>12} {'Davies-Bouldin':>16} {'Calinski-Harabasz':>18}")
print("-" * 80)
for name, res in ranked:
    sil = f"{res['silhouette']:.4f}" if not np.isnan(res['silhouette']) else "  N/A  "
    db  = f"{res['davies_bouldin']:.4f}" if not np.isnan(res['davies_bouldin']) else "  N/A  "
    ch  = f"{res['calinski_harabasz']:.1f}" if not np.isnan(res['calinski_harabasz']) else "  N/A  "
    print(f"{name:<18} {res['n_clusters']:>10} {sil:>12} {db:>16} {ch:>18}")

In [ ]:
# ---------------------------------------------------------------------------
# Bar chart: Silhouette scores
# ---------------------------------------------------------------------------
alg_names = [name for name, _ in ranked]
sil_scores = [res['silhouette'] for _, res in ranked]

colors = plt.cm.tab10(np.linspace(0, 0.5, len(alg_names)))

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(alg_names, sil_scores, color=colors, edgecolor='white', linewidth=0.8)
ax.axhline(0, color='black', linewidth=0.7, linestyle='--')
for bar, val in zip(bars, sil_scores):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.005, f"{val:.3f}",
                ha='center', va='bottom', fontsize=9)
ax.set_title("Silhouette Score Comparison (higher is better)")
ax.set_ylabel("Silhouette Score")
ax.set_ylim(bottom=min(0, min(v for v in sil_scores if not np.isnan(v))) - 0.05)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
_sil_path = Path(plots_dir) / "silhouette_comparison.png"
fig.savefig(_sil_path, dpi=120)
plt.show()
print(f"Saved: {_sil_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Multi-metric bar chart comparison
# ---------------------------------------------------------------------------
db_scores = [res['davies_bouldin'] for _, res in ranked]
ch_scores = [res['calinski_harabasz'] for _, res in ranked]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Silhouette (higher = better)
axes[0].bar(alg_names, sil_scores, color=colors, edgecolor='white')
axes[0].set_title("Silhouette Score\n(higher is better)")
axes[0].set_ylabel("Score")
axes[0].grid(axis='y', alpha=0.3)
axes[0].tick_params(axis='x', rotation=20)

# Davies-Bouldin (lower = better)
axes[1].bar(alg_names, db_scores, color=colors, edgecolor='white')
axes[1].set_title("Davies-Bouldin Score\n(lower is better)")
axes[1].set_ylabel("Score")
axes[1].grid(axis='y', alpha=0.3)
axes[1].tick_params(axis='x', rotation=20)

# Calinski-Harabasz (higher = better)
axes[2].bar(alg_names, ch_scores, color=colors, edgecolor='white')
axes[2].set_title("Calinski-Harabasz Score\n(higher is better)")
axes[2].set_ylabel("Score")
axes[2].grid(axis='y', alpha=0.3)
axes[2].tick_params(axis='x', rotation=20)

fig.tight_layout()
_multi_path = Path(plots_dir) / "metric_comparison.png"
fig.savefig(_multi_path, dpi=120)
plt.show()
print(f"Saved: {_multi_path}")

## 5 — Cluster Visualizations

In [ ]:
# ---------------------------------------------------------------------------
# UMAP cluster visualizations (best config per algorithm)
# ---------------------------------------------------------------------------

def _plot_clusters_2d(embedding: np.ndarray, labels: np.ndarray,
                      ax: plt.Axes, title: str) -> None:
    """Scatter plot of a 2D embedding colored by cluster labels.
    Noise points (label == -1) are shown as grey 'x' markers."""
    unique_labels = sorted(set(labels) - {-1})
    cmap = plt.cm.tab10
    norm = mcolors.BoundaryNorm(boundaries=range(len(unique_labels) + 1), ncolors=len(unique_labels))

    for idx, lbl in enumerate(unique_labels):
        mask = labels == lbl
        color = cmap(idx / max(len(unique_labels) - 1, 1))
        ax.scatter(embedding[mask, 0], embedding[mask, 1],
                   c=[color], s=8, alpha=0.6, linewidths=0, label=f"C{lbl}")

    # Noise
    noise_mask = labels == -1
    if noise_mask.any():
        ax.scatter(embedding[noise_mask, 0], embedding[noise_mask, 1],
                   c='grey', s=10, alpha=0.3, marker='x', linewidths=0.5, label='noise')

    n_shown = len(unique_labels)
    n_noise = int(noise_mask.sum())
    ax.set_title(f"{title}\n({n_shown} clusters, {n_noise} noise)")
    ax.set_xlabel("Dim-1")
    ax.set_ylabel("Dim-2")


alg_order = [name for name, _ in ranked]
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, name in zip(axes, alg_order):
    res = algorithm_results[name]
    _plot_clusters_2d(embedding_umap, res['labels'], ax, name)

fig.suptitle("UMAP Projections Colored by Cluster Assignment", fontsize=13, y=1.02)
fig.tight_layout()
_umap_cl_path = Path(plots_dir) / "umap_clusters.png"
fig.savefig(_umap_cl_path, dpi=120, bbox_inches='tight')
plt.show()
print(f"Saved: {_umap_cl_path}")

In [ ]:
# ---------------------------------------------------------------------------
# PCA cluster visualizations (best config per algorithm)
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, name in zip(axes, alg_order):
    res = algorithm_results[name]
    _plot_clusters_2d(embedding_pca, res['labels'], ax, name)

fig.suptitle("PCA Projections Colored by Cluster Assignment", fontsize=13, y=1.02)
fig.tight_layout()
_pca_cl_path = Path(plots_dir) / "pca_clusters.png"
fig.savefig(_pca_cl_path, dpi=120, bbox_inches='tight')
plt.show()
print(f"Saved: {_pca_cl_path}")

## 6 — Dendrogram: Visualizing Hierarchical Structure

Agglomerative clustering is a **hierarchical algorithm** — it builds a 
tree (dendrogram) by successively merging the closest clusters. The 
dendrogram reveals the **hierarchical structure** of your data:

- **Tall vertical bars** → large gap between clusters being merged → 
  natural cluster boundary at that level
- **The optimal cut** is at the level where merging distance jumps 
  sharply (the "elbow" in the dendrogram)
- **Ward linkage** minimizes the total within-cluster variance at each 
  merge step, producing compact, roughly equal-sized clusters

The dendrogram is the most informative diagnostic for agglomerative 
clustering: it shows all possible hierarchical solutions at once, not 
just a single choice of k.

In [ ]:
# ---------------------------------------------------------------------------
# Dendrogram — Ward Linkage on a subsample for readability
# ---------------------------------------------------------------------------
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist

# Use at most 200 points for readability (dendrograms become illegible with thousands)
rng_dendro = np.random.RandomState(random_state)
n_dendro = min(200, len(X))
dendro_idx = rng_dendro.choice(len(X), size=n_dendro, replace=False)
X_dendro = X[dendro_idx]

# Compute Ward linkage matrix
Z = linkage(X_dendro, method='ward', metric='euclidean')

# Determine horizontal cut levels from the last k merges
n_show = min(len(algorithm_results['Agglomerative']['labels']), 10)
last_merges = Z[-n_show:, 2]  # merge distances for last n_show merges

fig, (ax_dendro, ax_gap) = plt.subplots(1, 2, figsize=(16, 6))

# --- Full dendrogram (colour threshold at optimal k) ---
best_k_agg = algorithm_results['Agglomerative']['n_clusters']
# Find the linkage distance that cuts at best_k_agg clusters
cut_distance = float(Z[-(best_k_agg - 1), 2]) if best_k_agg > 1 else float(Z[-1, 2])

dendrogram(
    Z,
    ax=ax_dendro,
    truncate_mode='lastp',   # show only the last p merges
    p=50,                     # show the top-50 merges
    color_threshold=cut_distance,
    above_threshold_color='grey',
    leaf_rotation=90,
    leaf_font_size=8,
    no_labels=True,
)
ax_dendro.axhline(y=cut_distance, color='red', linestyle='--', linewidth=1.5,
                  label=f'Cut at k={best_k_agg} clusters')
ax_dendro.set_title(f'Dendrogram (Ward Linkage, subsample n={n_dendro})\n'
                     f'Red dashed line = optimal cut at k={best_k_agg}', fontsize=11)
ax_dendro.set_ylabel('Merge Distance (Ward)')
ax_dendro.legend(fontsize=9)
ax_dendro.grid(True, alpha=0.2, axis='y')

# --- Merge distance plot: identify the "elbow" ---
merge_indices = np.arange(1, len(Z) + 1)[::-1]
ax_gap.plot(range(1, n_show + 1), last_merges[::-1], 'o--', color='steelblue',
            linewidth=2, markersize=7)
ax_gap.set_xlabel('Number of clusters (from top of dendrogram)')
ax_gap.set_ylabel('Merge Distance')
ax_gap.set_title('Last Merge Distances\n(sharp jump = natural cluster boundary)', fontsize=11)
ax_gap.axvline(best_k_agg, color='red', linestyle='--', linewidth=1.5,
               label=f'Chosen k={best_k_agg}')
ax_gap.legend(fontsize=9)
ax_gap.grid(True, alpha=0.3)

plt.tight_layout()
_dendro_path = os.path.join(plots_dir, 'dendrogram.png')
fig.savefig(_dendro_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {_dendro_path}')
print(f'\nBest agglomerative k={best_k_agg} based on silhouette score.')
print('The dendrogram cut distance shows where Ward linkage sees the biggest natural gaps.')

## 7 — Per-Sample Silhouette Plot

The **silhouette score** is an aggregate — it averages over all samples.  
The **per-sample silhouette plot** shows the silhouette value for each 
individual point, sorted by cluster assignment:

- **Silhouette value ∈ [−1, 1]:**
  - Near +1 → point is well inside its cluster, far from neighbors
  - Near 0  → point is on the cluster boundary
  - Negative → point may be in the wrong cluster

A healthy clustering has **most bars positive and above the average**.  
Clusters with many negative values or bars much shorter than average 
suggest that cluster should be split or merged. The width of each cluster's 
section reflects its size — unequal widths indicate imbalanced clusters.

In [ ]:
# ---------------------------------------------------------------------------
# Per-Sample Silhouette Plot — KMeans (best silhouette scorer)
# ---------------------------------------------------------------------------
from sklearn.metrics import silhouette_samples

km_labels = algorithm_results['KMeans']['labels']
km_k = algorithm_results['KMeans']['n_clusters']

# Compute per-sample silhouette values
sample_silhouettes = silhouette_samples(X, km_labels)
avg_silhouette = float(sample_silhouettes.mean())

fig, ax = plt.subplots(figsize=(10, max(5, km_k * 1.5)))

y_lower = 10
cmap = plt.cm.tab10
cluster_stats = []

for cluster_id in range(km_k):
    mask = km_labels == cluster_id
    cluster_sil = np.sort(sample_silhouettes[mask])
    cluster_size = cluster_sil.shape[0]

    y_upper = y_lower + cluster_size
    color = cmap(cluster_id / max(km_k - 1, 1))

    ax.fill_betweenx(
        np.arange(y_lower, y_upper),
        0,
        cluster_sil,
        facecolor=color,
        edgecolor=color,
        alpha=0.7,
    )
    # Label each cluster at its vertical midpoint
    ax.text(-0.05, y_lower + cluster_size / 2, f'C{cluster_id}',
            fontsize=9, ha='right', va='center', color=color, fontweight='bold')

    cluster_stats.append({
        'cluster': cluster_id,
        'size': cluster_size,
        'mean_sil': float(cluster_sil.mean()),
        'n_negative': int((cluster_sil < 0).sum()),
    })
    y_lower = y_upper + 10  # gap between clusters

# Mean silhouette line
ax.axvline(avg_silhouette, color='red', linestyle='--', linewidth=2,
           label=f'Mean silhouette = {avg_silhouette:.3f}')
ax.axvline(0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)

ax.set_xlabel('Silhouette Coefficient', fontsize=11)
ax.set_ylabel('Cluster (sorted by silhouette within cluster)')
ax.set_title(f'Per-Sample Silhouette Plot — KMeans k={km_k}\n'
             f'Each cluster\'s width = its size; bars beyond red line = above-average quality',
             fontsize=11)
ax.set_yticks([])
ax.set_xlim(-0.3, 1.0)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2, axis='x')

plt.tight_layout()
_sil_path = os.path.join(plots_dir, 'per_sample_silhouette.png')
fig.savefig(_sil_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {_sil_path}')
print('\nCluster summary:')
for s in cluster_stats:
    print(f"  Cluster {s['cluster']}: size={s['size']}, mean_sil={s['mean_sil']:.3f}, "
          f"n_negative={s['n_negative']} ({100*s['n_negative']/max(s['size'],1):.1f}%)")

## 8 — GMM Ellipse Visualization

Gaussian Mixture Models fit **multivariate Gaussians** to the data —
each component has its own mean vector and covariance matrix.
Visualizing the **covariance ellipses** reveals what GMM actually 
learns:

- **Ellipse shape:** Diagonal covariance → axis-aligned ellipses.  
  Full covariance → rotated ellipses that capture feature correlations.
- **Ellipse size (2σ boundary):** ~95% of points from that Gaussian 
  should fall inside the 2σ ellipse.
- **Overlap:** Heavily overlapping ellipses mean the model sees 
  ambiguous cluster boundaries; low overlap means well-separated clusters.

Unlike KMeans (which forces spherical clusters), GMM can fit elongated,  
rotated clusters — the ellipses show you when this flexibility is used.

In [ ]:
# ---------------------------------------------------------------------------
# GMM Ellipse Visualization on PCA 2D Projection
# ---------------------------------------------------------------------------
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse

# Fit GMM on the PCA 2D embedding (so we can draw 2D ellipses)
best_k_gmm = algorithm_results['GMM']['n_clusters']
gmm_2d = GaussianMixture(n_components=best_k_gmm, covariance_type='full',
                          random_state=random_state, n_init=3)
gmm_2d.fit(embedding_pca)
gmm_labels_2d = gmm_2d.predict(embedding_pca)
gmm_proba_2d  = gmm_2d.predict_proba(embedding_pca)

def _draw_ellipse(mean, cov, ax, color, n_std=2.0, alpha=0.2):
    """Draw a confidence ellipse for a 2D Gaussian."""
    eigvals, eigvecs = np.linalg.eigh(cov)
    # Sort by eigenvalue descending
    order = eigvals.argsort()[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * n_std * np.sqrt(np.abs(eigvals))
    ell = Ellipse(xy=mean, width=width, height=height, angle=angle,
                  edgecolor=color, facecolor=color, linewidth=2,
                  alpha=alpha, zorder=2)
    ax.add_patch(ell)
    # Outer edge (no fill)
    ell_edge = Ellipse(xy=mean, width=width, height=height, angle=angle,
                       edgecolor=color, facecolor='none', linewidth=2.5, zorder=3)
    ax.add_patch(ell_edge)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cmap_gmm = plt.cm.tab10

# --- Left: scatter colored by cluster with ellipses ---
ax = axes[0]
for k in range(best_k_gmm):
    mask = gmm_labels_2d == k
    color = cmap_gmm(k / max(best_k_gmm - 1, 1))
    ax.scatter(embedding_pca[mask, 0], embedding_pca[mask, 1],
               c=[color], s=15, alpha=0.5, linewidths=0, zorder=1)
    _draw_ellipse(gmm_2d.means_[k], gmm_2d.covariances_[k], ax, color=color,
                  n_std=2.0, alpha=0.12)
    # Mark centroid
    ax.scatter(*gmm_2d.means_[k], marker='*', s=200, c=[color],
               edgecolors='black', linewidths=0.5, zorder=4)
ax.set_title(f'GMM k={best_k_gmm} — 2σ Covariance Ellipses on PCA Projection\n'
             f'Stars = component means, Ellipses = 95% probability contours', fontsize=10)
ax.set_xlabel('PCA-1')
ax.set_ylabel('PCA-2')
ax.grid(True, alpha=0.2)

# --- Right: uncertainty map — max posterior probability ---
ax = axes[1]
max_proba = gmm_proba_2d.max(axis=1)  # confidence of assignment
sc = ax.scatter(embedding_pca[:, 0], embedding_pca[:, 1],
                c=max_proba, cmap='RdYlGn', vmin=0.5, vmax=1.0,
                s=15, alpha=0.7, linewidths=0)
plt.colorbar(sc, ax=ax, label='Max posterior probability (assignment confidence)')
ax.set_title('GMM Assignment Confidence\n(green = confident, red = uncertain/boundary)',
             fontsize=10)
ax.set_xlabel('PCA-1')
ax.set_ylabel('PCA-2')
ax.grid(True, alpha=0.2)

plt.tight_layout()
_gmm_path = os.path.join(plots_dir, 'gmm_ellipses.png')
fig.savefig(_gmm_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {_gmm_path}')

# Component statistics
print(f'\nGMM Component Analysis (k={best_k_gmm}):')
print(f"{'Component':>9} | {'Weight':>8} | {'Mean norm':>10} | {'Avg confidence':>15}")
print('-' * 50)
for k in range(best_k_gmm):
    weight = float(gmm_2d.weights_[k])
    mean_norm = float(np.linalg.norm(gmm_2d.means_[k]))
    avg_conf = float(gmm_proba_2d[gmm_labels_2d == k, k].mean())
    print(f'{k:>9} | {weight:>8.3f} | {mean_norm:>10.3f} | {avg_conf:>15.3f}')
print('\nInterpretation:')
print('- Low avg_confidence (<0.7) → component boundary is diffuse, cluster may not be well-separated')
print('- Unequal weights → imbalanced clusters; consider if this reflects real structure')

## 6 — Metrics JSON Output

In [ ]:
# ---------------------------------------------------------------------------
# Write metrics JSON
# ---------------------------------------------------------------------------

def _safe_float(v):
    if v is None:
        return None
    try:
        f = float(v)
        return None if np.isnan(f) or np.isinf(f) else f
    except (TypeError, ValueError):
        return None


metrics_payload = {
    "run_metadata": {
        "timestamp": RUN_START.isoformat(),
        "n_samples": int(X.shape[0]),
        "n_features": int(X.shape[1]),
        "n_clusters_range": n_clusters_range,
        "dbscan_eps_range": dbscan_eps_range,
        "dbscan_min_samples": dbscan_min_samples,
        "random_state": random_state,
        "umap_n_neighbors": umap_n_neighbors,
        "synthetic_data": synthetic,
    },
    "results": {
        name: {
            "best_params": res['best_params'],
            "n_clusters": int(res['n_clusters']),
            "silhouette": _safe_float(res['silhouette']),
            "davies_bouldin": _safe_float(res['davies_bouldin']),
            "calinski_harabasz": _safe_float(res['calinski_harabasz']),
        }
        for name, res in algorithm_results.items()
    },
}

_metrics_path = Path(metrics_json_path)
_metrics_path.parent.mkdir(parents=True, exist_ok=True)
with open(_metrics_path, 'w') as fh:
    json.dump(metrics_payload, fh, indent=2)

print(f"Metrics written to: {_metrics_path}")
print(json.dumps(metrics_payload, indent=2))

## 7 — Summary

In [ ]:
# ---------------------------------------------------------------------------
# Final summary
# ---------------------------------------------------------------------------
RUN_END = datetime.now(timezone.utc)
RUN_DURATION = (RUN_END - RUN_START).total_seconds()

print("=" * 70)
print("CLUSTERING COMPARISON — FINAL SUMMARY")
print("=" * 70)
print(f"Run duration : {RUN_DURATION:.1f}s")
print(f"Dataset      : {X.shape[0]} samples, {X.shape[1]} features")
print(f"Synthetic    : {synthetic}")
print()
print(f"{'Rank':<6} {'Algorithm':<18} {'n_clusters':>10} {'Silhouette':>12} {'Davies-Bouldin':>16} {'Calinski-Harabasz':>18}")
print("-" * 84)
for rank, (name, res) in enumerate(ranked, 1):
    sil = f"{res['silhouette']:.4f}" if not np.isnan(res['silhouette']) else "  N/A  "
    db  = f"{res['davies_bouldin']:.4f}" if not np.isnan(res['davies_bouldin']) else "  N/A  "
    ch  = f"{res['calinski_harabasz']:.1f}" if not np.isnan(res['calinski_harabasz']) else "  N/A  "
    star = " <-- BEST" if rank == 1 else ""
    print(f"{rank:<6} {name:<18} {res['n_clusters']:>10} {sil:>12} {db:>16} {ch:>18}{star}")

best_name, _ = ranked[0]
print()
print(f"Best algorithm by silhouette: {best_name}")
print()
print("Output paths:")
print(f"  Metrics JSON  : {metrics_json_path}")
print(f"  Plots dir     : {plots_dir}")
if executed_notebook_path:
    print(f"  Notebook      : {executed_notebook_path}")
print("=" * 70)